# Axial buckling of integrally stringer-stiffened cylinders: Card and Jones (1966)

Card and Jones [1] tested in axial compression twelve cylinders with
longitudinal stiffeners, in pairs with the stiffeners outside (external) or
inside (internal) the skin, and compared the buckling loads with a theory of
eccentrically stiffened cylinders with the stiffeners averaged ("smeared")
over the skin, a membrane pre-buckling state, and simply supported or
clamped ends. Their cylinders 1 to 4 have integral rectangular (blade)
stiffeners machined from aluminium plates: cylinders 1 (external) and 2
(internal) are 38.00 in long, cylinders 3 (external) and 4 (internal) 23.75
in long. The externally stiffened cylinders carried about twice the load of
the internally stiffened ones.

Here the four cylinders are modelled with discrete stringers:

- the skin with the BFSC cylinder element of Sanders-type kinematics,
  `BFSCCylinderSanders`;
- each stringer with the BFSC plate element `BFSCPlate2D` of
  [bfscplate2d](https://github.com/saullocastro/bfscplate2d), in the
  plane $(x, r)$ of the stringer, from the middle surface of the skin to the
  tip, its root tied to the skin.

As in the notebook of the vibration tests of Sewall and Naumann, the model
is one stringer pitch with cyclic symmetry, given by the module
`blade_cylinder_sector.py` next to this notebook: one linear buckling problem
$(\boldsymbol{K}_{C0} + \lambda \boldsymbol{K}_G)\boldsymbol{\phi} =
\boldsymbol{0}$ per circumferential wave number $n$. Besides `bfsccylinder`,
it needs `bfscplate2d`, `composites`, SciPy and Matplotlib.

**Reference**

[1] Card, M. F.; Jones, R. M., "Experimental and theoretical results for
buckling of eccentrically stiffened cylinders", NASA TN D-3639, October
1966. https://ntrs.nasa.gov/citations/19660027866

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from bfsccylinder.version import __version__ as bfsccylinder_version
from bfscplate2d.version import __version__ as bfscplate2d_version
from blade_cylinder_sector import (BladeCylinderSector, buckling_membrane,
                                   prebuckling_clamped, buckling_prebuckled,
                                   axial_nodes, normalize_phase, W)

print('bfsccylinder', bfsccylinder_version)
print('bfscplate2d', bfscplate2d_version)

## Cylinders, geometry and material

Table I(a) of [1] (report page 7) gives, for each cylinder, the skin
thickness $t$ (the average of many micrometer measurements, individual
values scattering by ±8 %), the length $L$, the radius $R$ of the middle
surface, the area $A_t$ of the stiffened cross-section, the experimental
buckling load $\bar{P}_e$ and stress $\bar{\sigma}_e$, the number of
circumferential waves $n_e$ of the buckled test cylinder, and the axial
force per unit circumference at buckling, $\bar{N}_e$ from the test,
$\bar{N}_c$ from the theory with clamped ends and $\bar{N}_s$ from the
theory with simply supported ends. Its footnote gives $E = 10.5 \times 10^3$
ksi and $\mu = 0.32$ (2024-T351 aluminium alloy). Figure 2(b) of [1] (report
page 6) gives the stiffener: 0.097 in wide, at a spacing of 1.00 in, 0.33 in
from the surface of the skin opposite to the stiffener to the tip of the
stiffener, with fillets of radius 0.03 in; the spacing gives 60 stiffeners,
$2 \pi \times 9.55/1.00 = 60.0$.

In [ ]:
# Table I(a) of [1], report page 7; t, L, R in in, A_t in in^2, P_e in lbf,
# sigma_e in psi, N in lb/in (see below)
cylinders = {
    1: dict(side='external', t=0.0283, L=38.00, R=9.55, At=3.69, Pe=112.6e3,
            sigma_e=30.5e3, ne=6, Ne=1877., Nc=2036., Ns=1145.),
    2: dict(side='internal', t=0.0277, L=38.00, R=9.55, At=3.72, Pe=48.0e3,
            sigma_e=12.9e3, ne=6, Ne=800., Nc=997., Ns=687.),
    3: dict(side='external', t=0.0275, L=23.75, R=9.55, At=3.70, Pe=127.2e3,
            sigma_e=34.4e3, ne=7, Ne=2120., Nc=2595., Ns=1326.),
    4: dict(side='internal', t=0.0280, L=23.75, R=9.55, At=3.63, Pe=58.0e3,
            sigma_e=16.0e3, ne=6, Ne=965., Nc=1226., Ns=722.),
}
# Table I(b) of [1], report page 7: N_e, N_c, N_s as printed, header MN/m
si = {1: (329, 357, 200), 2: (140, 175, 120), 3: (371, 454, 232),
      4: (169, 215, 127)}
E = 10.5e6  # psi, footnote of Table I
mu = 0.32
nstringers = 60
bs = 0.097  # in, Figure 2(b)
depth = 0.33  # in, from the skin surface opposite to the stiffener
r_fillet = 0.03  # in

lbf = 4.4482216  # N
inch = 0.0254  # m
print('E = %.1f GPa' % (E*lbf/inch**2/1e9))
print('cyl.  P_e/(2 pi R)  N_e    N_e in kN/m  Table I(b)  '
      'P_e/A_t (sigma_e)')
for k, c in cylinders.items():
    print('%3d  %8.0f lb/in  %5.0f  %8.1f     %5d       %.1f ksi (%.1f)'
          % (k, c['Pe']/(2*np.pi*c['R']), c['Ne'],
             c['Ne']*lbf/inch/1e3, si[k][0], c['Pe']/c['At']/1e3,
             c['sigma_e']/1e3))

Two remarks on Table I of [1]:

- the columns $\bar{N}_e$, $\bar{N}_c$ and $\bar{N}_s$ of Table I(a) are
  headed kips/in., but their values are in lb/in: $\bar{P}_e/(2 \pi R)$
  gives $\bar{N}_e$ in lb/in, as printed above; likewise the values of Table
  I(b), headed MN/m, are the values of Table I(a) converted to kN/m, and
  $\bar{\sigma}_e = \bar{P}_e/A_t$;
- the area of the cross-section from the dimensions of Figure 2(b),
  computed below, is 5 % to 9 % smaller than $A_t$. The thicknesses $t$ were
  measured; the stiffener dimensions of Figure 2(b) are the drawing values,
  and the two butt joints of each cylinder, with two doubler plates 0.70 in
  wide and 0.020 in thick each ([1], report page 5), add only about 0.06
  in$^2$. The model follows Figure 2(b) and Table I and is not adjusted to
  $A_t$.

The plate of a stringer goes from the middle surface of the skin to the tip,
of height $H_s = 0.33 - t/2$ and thickness $b_s = 0.097$ in; the fillets
are not modelled.

In [ ]:
for k, c in cylinders.items():
    Hs = depth - c['t']/2
    fillets = nstringers*2*(1 - np.pi/4)*r_fillet**2
    drawn = 2*np.pi*c['R']*c['t'] + nstringers*bs*(depth - c['t']) + fillets
    model_area = 2*np.pi*c['R']*c['t'] + nstringers*bs*Hs
    print('cylinder %d: H_s = %.4f in; area from Figure 2(b) %.3f in^2 '
          '(%+.1f %% of A_t = %.2f), model %.3f in^2'
          % (k, Hs, drawn, 100*(drawn/c['At'] - 1), c['At'], model_area))

## Analyses

The buckling problems are solved for each harmonic $n = 0, 1, \ldots, 30$
of the sector of one stringer pitch: the modes of the whole cylinder with
$n$ circumferential waves, and with $n \pm 60 j$ waves coupled to them by
the discrete stringers, $n = 30$ including the local buckling of the skin
between the stringers with one half-wave per bay. The number of axial
half-waves $m$ of a mode is one more than the number of sign changes along
$x$ of the normal displacement $w$ of the skin. The critical load is given
as the average axial force per unit circumference $\bar{N} = \lambda
P/(2\pi R)$, as in [1].

Three analyses:

- **membrane, clamped**: the pre-buckling state of the theory of [1], a
  uniform axial strain $\epsilon$ without hoop stress, $N_{xx} = E t
  \epsilon$ in the skin and $E b_s \epsilon$ in the stringer, with the
  clamped ends of the theory, $u = v = w = w_{,x} = 0$ ([1], report page
  10), and the end edges of the stringers clamped; compared with
  $\bar{N}_c$. The geometric stiffness of the stringer includes the terms
  of its radial (in-plane) displacement, see `KG_inplane()` in
  `blade_cylinder_sector.py`: without them the axial force of the stringers
  would not act on the radial displacement of the stringers, which is the
  normal displacement $w$ of the skin;
- **membrane, simply supported**: the same pre-buckling state with the
  simply supported ends of the theory, $N_x = v = w = M_x = 0$ ([1], report
  page 9), i.e. $v = w = 0$ and $u$, $w_{,x}$ free, and $V = W = 0$ along
  the end edges of the stringers; compared with $\bar{N}_s$;
- **linear pre-buckling, clamped**: the tests idealized. The ends of the
  cylinders were potted, ground flat and parallel, and loaded by the platens
  of the testing machine, with aluminium bulkheads keeping them circular
  ([1], report page 6): the end $x = 0$ is clamped and the end $x = L$ is
  clamped with the prescribed axial displacement $u = U = -\delta$ of the
  skin and of the stringer. The axisymmetric pre-buckling state, with the
  radial displacement restrained at the ends, and the force $P$, the sum of
  the end reactions, come from a linear static analysis of the harmonic $n
  = 0$; compared with the test value $\bar{N}_e$.

In [ ]:
def build(k, nx, nbay, nh):
    c = cylinders[k]
    return BladeCylinderSector(R=c['R'], L=c['L'], h=c['t'], E=E, nu=mu,
                               rho=0., nstringers=nstringers, nx=nx,
                               nbay=nbay, bs=bs, Hs=depth - c['t']/2,
                               side=c['side'], nh=nh)


def scan(k, mesh, analysis, ns=range(0, 31), num=2):
    # lowest critical load and mode of each harmonic n
    model = build(k, *mesh)
    if analysis == 'linear':
        u0, P = prebuckling_clamped(model)
    out = {}
    for n in ns:
        if analysis == 'linear':
            N, X = buckling_prebuckled(model, u0, P, n, k=num)
        else:
            N, X = buckling_membrane(model, analysis, n, k=num)
        out[n] = (N[0], axial_nodes(model, X[:, 0]) + 1, X[:, 0])
    return model, out


def lowest(out):
    n = min(out, key=lambda n: out[n][0])
    return n, out[n]

## Mesh convergence

The membrane clamped analysis of the four cylinders with three meshes
`(nx, nbay, nh)`: `nx` nodes along $x$, `nbay` skin elements across the
stringer pitch and `nh` stringer elements along the height. The table gives
the lowest critical load over the harmonics $n = 3$ to $12$, which contain
the lowest modes, its $(m, n)$, the change from the previous mesh, and the
critical load of the harmonic $n = 30$ of the local modes of the skin
between the stringers.

In [ ]:
meshes = {23.75: [(17, 2, 2), (25, 4, 3), (37, 6, 4)],
          38.00: [(25, 2, 2), (37, 4, 3), (55, 6, 4)]}
conv = {}
t0 = time.time()
print('cyl.  mesh          dofs   N_cr (lb/in)  (m, n)   change    '
      'N_cr of n = 30')
for k, c in cylinders.items():
    prev = None
    for mesh in meshes[c['L']]:
        model, out = scan(k, mesh, 'CC', ns=list(range(3, 13)) + [30])
        conv[(k, mesh)] = (model, out)
        n, (N, m, _) = lowest(out)
        change = '' if prev is None else '%+.2f %%' % (100*(N/prev - 1))
        print('%3d   %-12s %5d   %8.0f      (%d, %2d)  %-8s  %8.0f'
              % (k, mesh, model.Nr, N, m, n, change, out[30][0]))
        prev = N
print('%.0f s' % (time.time() - t0))

The lowest critical loads change by less than 1 % between the two finest
meshes (see the table), which are used below.

## Critical loads

The three analyses of the four cylinders with the finest meshes, the lowest
critical load of each harmonic $n$:

In [ ]:
results = {}
t0 = time.time()
for k, c in cylinders.items():
    mesh = meshes[c['L']][-1]
    for analysis in ['CC', 'SS', 'linear']:
        results[(k, analysis)] = scan(k, mesh, analysis)
print('%.0f s' % (time.time() - t0))

names = {'CC': 'membrane, clamped', 'SS': 'membrane, simply supp.',
         'linear': 'linear pre-buckling, clamped'}
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
for ax, (k, c) in zip(axes.ravel(), cylinders.items()):
    for analysis, style in [('CC', 'C0-'), ('SS', 'C1-'),
                            ('linear', 'C2--')]:
        out = results[(k, analysis)][1]
        ns = sorted(out)
        ax.plot(ns, [out[n][0] for n in ns], style, label=names[analysis])
    for key, style, label in [('Nc', 'C0:', r'$\bar{N}_c$ [1]'),
                              ('Ns', 'C1:', r'$\bar{N}_s$ [1]'),
                              ('Ne', 'k:', r'$\bar{N}_e$ test [1]')]:
        ax.axhline(c[key], ls=':', color=style[:-1], lw=1.2, label=label)
    ax.plot([c['ne']], [c['Ne']], 'ko', mfc='none')
    ax.set_ylim(0, 2.5*c['Nc'])
    ax.set_title('cylinder %d, %s, L = %.2f in' % (k, c['side'], c['L']),
                 fontsize=10)
    ax.grid(True, lw=0.3)
for ax in axes[1]:
    ax.set_xlabel('harmonic $n$')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$\bar{N}_{cr}$ (lb/in)')
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Comparison with the theory and the tests of Card and Jones

The relative difference is (FE − [1])/[1].

In [ ]:
print('cyl.  side      analysis                      FE (lb/in) (m, n)    '
      '[1] (lb/in)            diff')
for k, c in cylinders.items():
    for analysis, key, label in [('CC', 'Nc', 'N_c, theory'),
                                 ('SS', 'Ns', 'N_s, theory'),
                                 ('linear', 'Ne', 'N_e, test')]:
        n, (N, m, _) = lowest(results[(k, analysis)][1])
        print('%3d   %-8s  %-29s %7.0f   (%d, %2d)   %5.0f %-14s %+6.1f %%'
              % (k, c['side'], names[analysis], N, m, n, c[key], label,
                 100*(N/c[key] - 1)))

print('\nTest wave numbers n_e and the linear pre-buckling analysis:')
print('cyl.  n_e  N_cr at n_e (lb/in)  lowest N_cr (n)   N_e (lb/in)  '
      'N_e/N_cr(lowest)')
for k, c in cylinders.items():
    out = results[(k, 'linear')][1]
    n, (N, m, _) = lowest(out)
    print('%3d   %3d   %10.0f           %7.0f (%2d)      %6.0f       %.2f'
          % (k, c['ne'], out[c['ne']][0], N, n, c['Ne'], c['Ne']/N))

print('\nExternal over internal stiffeners, ratio of the lowest loads:')
for pair in [(1, 2), (3, 4)]:
    row = []
    for analysis, key in [('CC', 'Nc'), ('SS', 'Ns'), ('linear', 'Ne')]:
        fe = [lowest(results[(k, analysis)][1])[1][0] for k in pair]
        ref = [cylinders[k][key] for k in pair]
        row.append('%s: FE %.2f, [1] %.2f' % (key, fe[0]/fe[1],
                                               ref[0]/ref[1]))
    print('cylinders %d/%d: ' % pair + '; '.join(row))

## Buckling modes

The normal displacement $w$ of the skin over the developed cylinder, rebuilt
from the sector, for the lowest mode of the linear pre-buckling analysis of
each cylinder, and the lowest mode of the harmonic $n = 30$ of cylinder 3,
the local buckling of the skin between the stringers:

In [ ]:
def plot_w(ax, model, mode, n, title):
    w = normalize_phase(model.skin_field(mode, W))
    theta, wfull = model.full_circumference(w, n)
    lim = np.abs(wfull).max()
    ax.contourf(np.degrees(theta), model.xlin, wfull,
                levels=np.linspace(-lim, lim, 31), cmap='RdBu_r')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel(r'$\theta$ (degrees)')


fig, axes = plt.subplots(1, 5, figsize=(15, 3.8))
for ax, k in zip(axes, cylinders):
    model, out = results[(k, 'linear')]
    n, (N, m, mode) = lowest(out)
    plot_w(ax, model, mode, n, 'cylinder %d, (m, n) = (%d, %d), %.0f lb/in'
           % (k, m, n, N))
model, out = results[(3, 'linear')]
N, m, mode = out[30]
plot_w(axes[4], model, mode, 30, 'cylinder 3, n = 30, %.0f lb/in' % N)
axes[4].set_xlim(0, 60)
axes[0].set_ylabel('$x$ (in)')
plt.tight_layout()
plt.show()

## Discussion

TODO_DISCUSSION